In [8]:
# feature_engineer.py
"""
Advanced Feature Engineering for Multi-Disease Prediction Project
CS-245 Course Project: Health & Public Safety Intelligence Theme
Creates sophisticated features from BRFSS and EPA data for predictive modeling
"""

import pandas as pd
import numpy as np
import os
import logging
from typing import List, Dict, Tuple, Optional
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import warnings
warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class MultiDiseaseFeatureEngineer:
    """
    Advanced feature engineering pipeline for multi-disease prediction.
    Creates clinical, behavioral, environmental, and composite features.
    """
    
    def __init__(self, data_path: str):
        """
        Initialize the feature engineer with dataset path.
        
        Args:
            data_path: Path to the merged dataset from data_merge.py
        """
        self.data_path = data_path
        self.df = None
        self.features = None
        self.targets = None
        self.scalers = {}
        self.feature_categories = {}
        
    def load_data(self) -> pd.DataFrame:
        """Load and validate the merged dataset."""
        logger.info(f"Loading data from {self.data_path}")
        
        if not os.path.exists(self.data_path):
            logger.error(f"Data file not found: {self.data_path}")
            raise FileNotFoundError(f"Data file not found: {self.data_path}")
        
        self.df = pd.read_csv(self.data_path)
        logger.info(f"Loaded dataset: {self.df.shape[0]} counties, {self.df.shape[1]} features")
        
        # Basic data validation
        self._validate_data()
        
        return self.df
    
    def check_data_quality(self):
        """Check for data quality issues."""
        if self.df is None:
            self.load_data()
        
        logger.info("Checking data quality...")
        
        # Check for NaN values
        nan_counts = self.df.isna().sum()
        nan_cols = nan_counts[nan_counts > 0]
        
        if len(nan_cols) > 0:
            logger.warning(f"Found {len(nan_cols)} columns with NaN values:")
            for col, count in nan_cols.items():
                logger.warning(f"  {col}: {count} NaN values ({count/len(self.df):.1%})")
        
        # Check for infinite values
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if np.isinf(self.df[col]).any():
                logger.warning(f"Column {col} contains infinite values")
        
        # Check for constant columns
        constant_cols = []
        for col in numeric_cols:
            if self.df[col].nunique() <= 1:
                constant_cols.append(col)
        
        if constant_cols:
            logger.warning(f"Found {len(constant_cols)} constant columns: {constant_cols}")
        
        return {
            'nan_columns': nan_cols,
            'constant_columns': constant_cols
        }
        
    def _validate_data(self):
        """Validate the dataset structure and required columns."""
        # Check for expected column patterns
        expected_patterns = ['Pct_', 'Avg_', 'State_']
        found_patterns = []
        
        for pattern in expected_patterns:
            pattern_cols = [col for col in self.df.columns if col.startswith(pattern)]
            if pattern_cols:
                found_patterns.append(pattern)
                logger.info(f"Found {len(pattern_cols)} columns with pattern '{pattern}'")
        
        if not found_patterns:
            logger.warning("Dataset may not have expected feature structure")
        else:
            logger.info(f"Found columns with patterns: {', '.join(found_patterns)}")
        
        # Check for target columns
        disease_targets = ['Pct_Diabetes', 'Pct_HeartDisease', 'Pct_Stroke', 'Pct_Asthma']
        self.targets = []
        
        for target in disease_targets:
            if target in self.df.columns:
                # Check if target has valid data
                if self.df[target].notna().any():
                    self.targets.append(target)
                else:
                    logger.warning(f"Target column {target} exists but has no valid data")
            else:
                logger.warning(f"Target column {target} not found in dataset")
        
        if not self.targets:
            logger.warning("No disease prevalence targets found in dataset with valid data")
        else:
            logger.info(f"Found {len(self.targets)} disease targets with valid data: {', '.join(self.targets)}")
            
            # Log some statistics about the targets
            for target in self.targets:
                target_data = self.df[target].dropna()
                if len(target_data) > 0:
                    logger.info(f"  {target}: {len(target_data)} valid values, "
                               f"mean={target_data.mean():.2f}, range=[{target_data.min():.2f}, {target_data.max():.2f}]")
    
    def create_clinical_risk_features(self) -> pd.DataFrame:
        """
        Create clinical risk assessment features based on established medical guidelines.
        
        Returns:
            DataFrame with clinical risk features
        """
        logger.info("Creating clinical risk assessment features...")
        
        clinical_features = pd.DataFrame(index=self.df.index)
        
        # 1. Framingham-like Cardiovascular Risk Score (simplified)
        cv_risk_factors = []
        if 'Pct_HighBP' in self.df.columns:
            clinical_features['HighBP_Risk'] = self.df['Pct_HighBP'] / 100
            cv_risk_factors.append('HighBP_Risk')
        
        if 'Pct_HighChol' in self.df.columns:
            clinical_features['HighChol_Risk'] = self.df['Pct_HighChol'] / 100
            cv_risk_factors.append('HighChol_Risk')
        
        if 'Pct_Smoker' in self.df.columns:
            clinical_features['Smoking_Risk'] = self.df['Pct_Smoker'] / 100
            cv_risk_factors.append('Smoking_Risk')
        
        if 'Pct_Diabetes' in self.df.columns:
            clinical_features['Diabetes_Risk'] = self.df['Pct_Diabetes'] / 100
            cv_risk_factors.append('Diabetes_Risk')
        
        if cv_risk_factors:
            clinical_features['Cardiovascular_Risk_Index'] = clinical_features[cv_risk_factors].mean(axis=1)
        
        # 2. Metabolic Syndrome Indicator
        metabolic_factors = []
        if 'Avg_BMI' in self.df.columns:
            # BMI > 30 indicates obesity
            clinical_features['Obesity_Indicator'] = (self.df['Avg_BMI'] > 30).astype(int)
            metabolic_factors.append('Obesity_Indicator')
        
        if 'Pct_HighBP' in self.df.columns:
            clinical_features['Hypertension_Indicator'] = (self.df['Pct_HighBP'] > 35).astype(int)
            metabolic_factors.append('Hypertension_Indicator')
        
        if 'Pct_HighChol' in self.df.columns:
            clinical_features['HighChol_Indicator'] = (self.df['Pct_HighChol'] > 35).astype(int)
            metabolic_factors.append('HighChol_Indicator')
        
        if metabolic_factors:
            clinical_features['Metabolic_Syndrome_Score'] = clinical_features[metabolic_factors].sum(axis=1)
        
        # 3. Respiratory Risk Score
        respiratory_factors = []
        if 'Pct_Smoker' in self.df.columns:
            clinical_features['Smoking_Resp_Risk'] = self.df['Pct_Smoker'] / 100
            respiratory_factors.append('Smoking_Resp_Risk')
        
        if 'State_Avg_AQI' in self.df.columns:
            # Normalize AQI to 0-1 risk scale (AQI > 100 is unhealthy)
            aqi_risk = (self.df['State_Avg_AQI'] - 50) / 150  # Assuming 50-200 range
            clinical_features['AQI_Resp_Risk'] = np.clip(aqi_risk, 0, 1)
            respiratory_factors.append('AQI_Resp_Risk')
        
        if respiratory_factors:
            clinical_features['Respiratory_Risk_Score'] = clinical_features[respiratory_factors].mean(axis=1)
        
        # 4. Comorbidity Interaction Features
        if all(col in self.df.columns for col in ['Pct_Diabetes', 'Pct_HighBP']):
            clinical_features['Diabetes_Hypertension_Interaction'] = (
                self.df['Pct_Diabetes'] * self.df['Pct_HighBP'] / 10000
            )
        
        if all(col in self.df.columns for col in ['Pct_HeartDisease', 'Pct_Diabetes']):
            clinical_features['CVD_Diabetes_Interaction'] = (
                self.df['Pct_HeartDisease'] * self.df['Pct_Diabetes'] / 10000
            )
        
        logger.info(f"Created {len(clinical_features.columns)} clinical risk features")
        return clinical_features
    
    def create_behavioral_lifestyle_features(self) -> pd.DataFrame:
        """
        Create behavioral and lifestyle-based features.
        
        Returns:
            DataFrame with behavioral lifestyle features
        """
        logger.info("Creating behavioral lifestyle features...")
        
        behavioral_features = pd.DataFrame(index=self.df.index)
        
        # 1. Healthy Lifestyle Score
        healthy_factors = []
        
        if 'Pct_PhysicalActivity' in self.df.columns:
            behavioral_features['Physical_Activity_Score'] = self.df['Pct_PhysicalActivity'] / 100
            healthy_factors.append('Physical_Activity_Score')
        
        if 'Avg_HealthyDiet_Score' in self.df.columns:
            behavioral_features['Diet_Quality_Score'] = self.df['Avg_HealthyDiet_Score']
            healthy_factors.append('Diet_Quality_Score')
        
        if 'Pct_Smoker' in self.df.columns:
            # Inverted: non-smoking is healthy
            behavioral_features['Non_Smoking_Score'] = 1 - (self.df['Pct_Smoker'] / 100)
            healthy_factors.append('Non_Smoking_Score')
        
        if 'Pct_HeavyDrinker' in self.df.columns:
            # Inverted: non-heavy drinking is healthy
            behavioral_features['Moderate_Drinking_Score'] = 1 - (self.df['Pct_HeavyDrinker'] / 100)
            healthy_factors.append('Moderate_Drinking_Score')
        
        if healthy_factors:
            behavioral_features['Healthy_Lifestyle_Index'] = behavioral_features[healthy_factors].mean(axis=1)
        
        # 2. Risk Behavior Clusters
        risk_factors = []
        if 'Pct_Smoker' in self.df.columns:
            behavioral_features['Smoking_Risk'] = self.df['Pct_Smoker'] / 100
            risk_factors.append('Smoking_Risk')
        
        if 'Pct_HeavyDrinker' in self.df.columns:
            behavioral_features['Alcohol_Risk'] = self.df['Pct_HeavyDrinker'] / 100
            risk_factors.append('Alcohol_Risk')
        
        if 'Avg_SleepHours' in self.df.columns and self.df['Avg_SleepHours'].notna().any():
            # Both too little and too much sleep are risky
            sleep_risk = np.abs(self.df['Avg_SleepHours'] - 7) / 7  # 7 hours is optimal
            behavioral_features['Sleep_Risk'] = np.clip(sleep_risk, 0, 1)
            risk_factors.append('Sleep_Risk')
        
        if risk_factors:
            behavioral_features['Behavioral_Risk_Score'] = behavioral_features[risk_factors].mean(axis=1)
        
        # 3. Healthcare Access Composite
        if 'Pct_AnyHealthcare' in self.df.columns:
            # Check if column has valid data
            if self.df['Pct_AnyHealthcare'].notna().any():
                behavioral_features['Healthcare_Access_Score'] = self.df['Pct_AnyHealthcare'] / 100
            else:
                logger.warning("Pct_AnyHealthcare column has no valid data, skipping...")
        
        # 4. Preventive Care Score
        if 'Pct_CholCheck' in self.df.columns:
            behavioral_features['Preventive_Care_Score'] = self.df['Pct_CholCheck'] / 100
        
        # 5. Sedentary Lifestyle Indicator
        if 'Avg_TotalUnhealthyDays' in self.df.columns:
            # More unhealthy days indicates more sedentary lifestyle
            behavioral_features['Sedentary_Indicator'] = self.df['Avg_TotalUnhealthyDays'] / 30
        
        logger.info(f"Created {len(behavioral_features.columns)} behavioral lifestyle features")
        return behavioral_features
    
    def create_environmental_features(self) -> pd.DataFrame:
        """
        Create environmental and geographic features.
        
        Returns:
            DataFrame with environmental features
        """
        logger.info("Creating environmental features...")
        
        env_features = pd.DataFrame(index=self.df.index)
        
        # 1. Air Quality Risk Tiers
        if 'State_Avg_AQI' in self.df.columns:
            # Categorize AQI levels
            aqi_bins = [0, 50, 100, 150, 200, 300, 1000]
            aqi_labels = ['Good', 'Moderate', 'Unhealthy_Sensitive', 'Unhealthy', 'Very_Unhealthy', 'Hazardous']
            
            env_features['AQI_Category'] = pd.cut(
                self.df['State_Avg_AQI'],
                bins=aqi_bins,
                labels=aqi_labels,
                include_lowest=True
            )
            
            # One-hot encode AQI categories
            aqi_dummies = pd.get_dummies(env_features['AQI_Category'], prefix='AQI')
            env_features = pd.concat([env_features, aqi_dummies], axis=1)
            
            # Create AQI severity score
            aqi_severity_map = {
                'Good': 1,
                'Moderate': 2,
                'Unhealthy_Sensitive': 3,
                'Unhealthy': 4,
                'Very_Unhealthy': 5,
                'Hazardous': 6
            }
            env_features['AQI_Severity_Score'] = env_features['AQI_Category'].map(aqi_severity_map)
        
        # 2. Pollution Exposure Duration
        if 'State_Unhealthy_Days_Pct' in self.df.columns:
            env_features['Chronic_Pollution_Exposure'] = self.df['State_Unhealthy_Days_Pct'] / 100
        
        # 3. Environmental Justice Index (combining pollution and health vulnerability)
        if all(col in self.df.columns for col in ['State_Avg_AQI', 'Pct_Diabetes']):
            # Normalize both to 0-1 scale
            aqi_norm = (self.df['State_Avg_AQI'] - self.df['State_Avg_AQI'].min()) / \
                      (self.df['State_Avg_AQI'].max() - self.df['State_Avg_AQI'].min())
            diabetes_norm = (self.df['Pct_Diabetes'] - self.df['Pct_Diabetes'].min()) / \
                           (self.df['Pct_Diabetes'].max() - self.df['Pct_Diabetes'].min())
            
            env_features['Environmental_Justice_Risk'] = (aqi_norm + diabetes_norm) / 2
        
        # 4. Seasonal Air Quality Variation (simulated)
        # Assuming higher AQI in summer months
        if 'State' in self.df.columns:
            # Create region-based features
            southern_states = ['Texas', 'Florida', 'Georgia', 'Alabama', 'Mississippi', 
                              'Louisiana', 'South Carolina', 'North Carolina']
            env_features['Southern_Region'] = self.df['State'].isin(southern_states).astype(int)
            
            western_states = ['California', 'Arizona', 'Nevada', 'Utah', 'Colorado', 
                             'New Mexico', 'Oregon', 'Washington']
            env_features['Western_Region'] = self.df['State'].isin(western_states).astype(int)
        
        logger.info(f"Created {len(env_features.columns)} environmental features")
        return env_features
    
    def create_demographic_features(self) -> pd.DataFrame:
        """
        Create demographic and socioeconomic features.
        
        Returns:
            DataFrame with demographic features
        """
        logger.info("Creating demographic features...")
        
        demo_features = pd.DataFrame(index=self.df.index)
        
        # 1. Age Distribution Features
        if 'Avg_Age' in self.df.columns:
            # Age groups risk factors
            demo_features['Senior_Population_Risk'] = (self.df['Avg_Age'] > 65).astype(int)
            demo_features['Working_Age_Population'] = ((self.df['Avg_Age'] >= 18) & (self.df['Avg_Age'] <= 64)).astype(int)
            
            # Age risk curve (higher risk for older populations)
            demo_features['Age_Risk_Score'] = np.where(
                self.df['Avg_Age'] < 40,
                self.df['Avg_Age'] / 200,
                (self.df['Avg_Age'] - 40) / 100 + 0.2
            )
        
        # 2. Sample Size Weighted Features
        if 'Sample_Size' in self.df.columns:
            demo_features['Data_Quality_Score'] = np.log1p(self.df['Sample_Size']) / np.log1p(self.df['Sample_Size'].max())
        
        # 3. Population Density Proxy (using sample size as proxy)
        if 'Sample_Size' in self.df.columns:
            demo_features['Population_Density_Proxy'] = self.df['Sample_Size'] / self.df['Sample_Size'].max()
        
        # 4. Urban-Rural Indicator (simplified)
        if all(col in self.df.columns for col in ['Sample_Size', 'State']):
            # Large sample sizes might indicate urban areas
            demo_features['Urban_Indicator'] = (self.df['Sample_Size'] > self.df['Sample_Size'].median()).astype(int)
        
        logger.info(f"Created {len(demo_features.columns)} demographic features")
        return demo_features
    
    def create_composite_risk_scores(self) -> pd.DataFrame:
        """
        Create comprehensive composite risk scores combining multiple domains.
        
        Returns:
            DataFrame with composite risk scores
        """
        logger.info("Creating composite risk scores...")
        
        composite_features = pd.DataFrame(index=self.df.index)
        
        # 1. Overall Health Risk Score (combines clinical, behavioral, environmental)
        risk_components = []
        
        # Clinical component
        if 'Cardiovascular_Risk_Index' in self.df.columns:
            composite_features['Clinical_Risk_Component'] = self.df['Cardiovascular_Risk_Index']
            risk_components.append('Clinical_Risk_Component')
        
        # Behavioral component
        if 'Behavioral_Risk_Score' in self.df.columns:
            composite_features['Behavioral_Risk_Component'] = self.df['Behavioral_Risk_Score']
            risk_components.append('Behavioral_Risk_Component')
        
        # Environmental component
        if 'AQI_Severity_Score' in self.df.columns:
            env_norm = (self.df['AQI_Severity_Score'] - 1) / 5  # Normalize 1-6 to 0-1
            composite_features['Environmental_Risk_Component'] = env_norm
            risk_components.append('Environmental_Risk_Component')
        
        if risk_components:
            composite_features['Overall_Health_Risk_Score'] = composite_features[risk_components].mean(axis=1)
        
        # 2. Chronic Disease Burden Index
        chronic_diseases = [col for col in self.df.columns if col.startswith('Pct_') and 
                           any(disease in col for disease in ['Diabetes', 'Heart', 'Stroke', 'COPD', 'Kidney'])]
        
        if chronic_diseases:
            # Calculate weighted average of chronic disease prevalences
            disease_weights = {
                'Diabetes': 1.2,  # Higher weight for diabetes
                'Heart': 1.5,     # Highest weight for heart disease
                'Stroke': 1.3,    # High weight for stroke
                'COPD': 1.1,      # Moderate weight for COPD
                'Kidney': 1.1     # Moderate weight for kidney disease
            }
            
            weighted_sum = 0
            total_weight = 0
            
            for disease_col in chronic_diseases:
                for disease_name, weight in disease_weights.items():
                    if disease_name in disease_col:
                        weighted_sum += self.df[disease_col] * weight
                        total_weight += weight
                        break
            
            if total_weight > 0:
                composite_features['Chronic_Disease_Burden_Index'] = weighted_sum / total_weight
        
        # 3. Preventable Disease Risk Score
        preventable_factors = []
        
        if 'Pct_Smoker' in self.df.columns:
            composite_features['Smoking_Preventable_Risk'] = self.df['Pct_Smoker'] / 100
            preventable_factors.append('Smoking_Preventable_Risk')
        
        if 'Pct_HighBP' in self.df.columns:
            composite_features['Hypertension_Preventable_Risk'] = self.df['Pct_HighBP'] / 100 * 0.8  # 80% preventable
            preventable_factors.append('Hypertension_Preventable_Risk')
        
        if preventable_factors:
            composite_features['Preventable_Disease_Risk_Score'] = composite_features[preventable_factors].mean(axis=1)
        
        # 4. Health Equity Score (combining healthcare access and environmental justice)
        equity_factors = []
        
        if 'Healthcare_Access_Score' in self.df.columns:
            # Invert so lower access = higher inequity
            composite_features['Healthcare_Inequity'] = 1 - self.df['Healthcare_Access_Score']
            equity_factors.append('Healthcare_Inequity')
        
        if 'Environmental_Justice_Risk' in self.df.columns:
            composite_features['Environmental_Inequity'] = self.df['Environmental_Justice_Risk']
            equity_factors.append('Environmental_Inequity')
        
        if equity_factors:
            composite_features['Health_Equity_Score'] = 1 - composite_features[equity_factors].mean(axis=1)
        
        logger.info(f"Created {len(composite_features.columns)} composite risk scores")
        return composite_features
    
    def create_interaction_features(self) -> pd.DataFrame:
        """
        Create interaction features between different risk domains.
        
        Returns:
            DataFrame with interaction features
        """
        logger.info("Creating interaction features...")
        
        interaction_features = pd.DataFrame(index=self.df.index)
        
        # 1. Age-Environment Interactions
        if all(col in self.df.columns for col in ['Avg_Age', 'State_Avg_AQI']):
            interaction_features['Age_AQI_Interaction'] = (
                self.df['Avg_Age'] * self.df['State_Avg_AQI'] / 100
            )
        
        # 2. Smoking-Environment Interactions
        if all(col in self.df.columns for col in ['Pct_Smoker', 'State_Avg_AQI']):
            interaction_features['Smoking_AQI_Interaction'] = (
                self.df['Pct_Smoker'] * self.df['State_Avg_AQI'] / 10000
            )
        
        # 3. Poverty-Health Interactions (using sample size as proxy for economic status)
        if all(col in self.df.columns for col in ['Sample_Size', 'Pct_Diabetes']):
            # Smaller sample size might indicate rural/poorer areas
            poverty_proxy = 1 / np.log1p(self.df['Sample_Size'])
            poverty_proxy_norm = (poverty_proxy - poverty_proxy.min()) / \
                               (poverty_proxy.max() - poverty_proxy.min())
            
            interaction_features['Poverty_Diabetes_Interaction'] = (
                poverty_proxy_norm * self.df['Pct_Diabetes'] / 100
            )
        
        # 4. Multi-Risk Amplification Features
        if all(col in self.df.columns for col in ['Pct_Smoker', 'Pct_HighBP', 'Pct_Diabetes']):
            interaction_features['Triple_Risk_Amplification'] = (
                (self.df['Pct_Smoker'] / 100) *
                (self.df['Pct_HighBP'] / 100) *
                (self.df['Pct_Diabetes'] / 100)
            )
        
        # 5. Lifestyle-Environment Synergy
        if all(col in self.df.columns for col in ['Pct_PhysicalActivity', 'State_Avg_AQI']):
            # Physical activity might mitigate poor air quality effects
            interaction_features['Exercise_AQI_Protection'] = (
                (self.df['Pct_PhysicalActivity'] / 100) *
                (1 / (self.df['State_Avg_AQI'] / 100 + 0.1))  # Higher AQI reduces protection
            )
        
        logger.info(f"Created {len(interaction_features.columns)} interaction features")
        return interaction_features
    
    def create_statistical_features(self) -> pd.DataFrame:
        """
        Create statistical and derived mathematical features.
        
        Returns:
            DataFrame with statistical features
        """
        logger.info("Creating statistical features...")
        
        stats_features = pd.DataFrame(index=self.df.index)
        
        # 1. Disease Prevalence Ratios
        if all(col in self.df.columns for col in ['Pct_Diabetes', 'Pct_HighBP']):
            stats_features['Diabetes_to_Hypertension_Ratio'] = (
                self.df['Pct_Diabetes'] / (self.df['Pct_HighBP'] + 1e-10)
            )
        
        if all(col in self.df.columns for col in ['Pct_HeartDisease', 'Pct_Diabetes']):
            stats_features['CVD_to_Diabetes_Ratio'] = (
                self.df['Pct_HeartDisease'] / (self.df['Pct_Diabetes'] + 1e-10)
            )
        
        # 2. Risk Factor Aggregation
        risk_factor_cols = [col for col in self.df.columns if col.startswith('Pct_') and 
                           any(factor in col for factor in ['Smoker', 'HighBP', 'HighChol', 'HeavyDrinker'])]
        
        if risk_factor_cols:
            stats_features['Risk_Factor_Count'] = (self.df[risk_factor_cols] > 30).sum(axis=1)
            stats_features['Risk_Factor_Mean'] = self.df[risk_factor_cols].mean(axis=1)
            stats_features['Risk_Factor_Std'] = self.df[risk_factor_cols].std(axis=1)
        
        # 3. Disease Co-occurrence Patterns
        disease_cols = [col for col in self.df.columns if col.startswith('Pct_') and 
                       any(disease in col for disease in ['Diabetes', 'HeartDisease', 'Stroke', 'Asthma', 'COPD'])]
        
        if len(disease_cols) >= 2:
            # Calculate correlation-like measure between diseases within county
            # This is simplified - in reality would need individual-level data
            stats_features['Disease_Correlation_Proxy'] = self.df[disease_cols].std(axis=1) / \
                                                         (self.df[disease_cols].mean(axis=1) + 1e-10)
        
        # 4. Outlier Detection Features
        if 'Pct_Diabetes' in self.df.columns:
            diabetes_zscore = (self.df['Pct_Diabetes'] - self.df['Pct_Diabetes'].mean()) / \
                            self.df['Pct_Diabetes'].std()
            stats_features['Diabetes_Outlier_Score'] = np.abs(diabetes_zscore)
        
        # 5. Trend Features (if we had temporal data)
        # For now, create regional comparison features
        if 'State' in self.df.columns and 'Pct_Diabetes' in self.df.columns:
            state_avg_diabetes = self.df.groupby('State')['Pct_Diabetes'].transform('mean')
            stats_features['Diabetes_vs_State_Avg'] = self.df['Pct_Diabetes'] - state_avg_diabetes
        
        logger.info(f"Created {len(stats_features.columns)} statistical features")
        return stats_features
    
    def create_clustering_features(self, n_clusters: int = 5) -> pd.DataFrame:
        """
        Create clustering-based features to identify county types.
        
        Args:
            n_clusters: Number of clusters for K-means
            
        Returns:
            DataFrame with clustering features
        """
        logger.info(f"Creating clustering features with {n_clusters} clusters...")
        
        clustering_features = pd.DataFrame(index=self.df.index)
        
        # Select features for clustering
        cluster_features = []
        
        # Key features for clustering
        candidate_features = [
            'Pct_Diabetes', 'Pct_HeartDisease', 'Pct_Stroke', 'Pct_Asthma',
            'Pct_Smoker', 'Pct_HighBP', 'State_Avg_AQI', 'Avg_Age'
        ]
        
        cluster_features = [col for col in candidate_features if col in self.df.columns]
        
        if len(cluster_features) < 3:
            logger.warning(f"Insufficient features for clustering. Need at least 3, found {len(cluster_features)}")
            return clustering_features
        
        # Prepare data for clustering
        X_cluster = self.df[cluster_features].copy()
        
        # Check for NaN values
        nan_cols = X_cluster.columns[X_cluster.isna().any()].tolist()
        if nan_cols:
            logger.info(f"Found {len(nan_cols)} columns with NaN values for clustering. Filling with column medians...")
            
            # Check if any column has all NaN values
            all_nan_cols = X_cluster.columns[X_cluster.isna().all()].tolist()
            if all_nan_cols:
                logger.warning(f"Columns with all NaN values for clustering: {all_nan_cols}. These will be dropped.")
                X_cluster = X_cluster.drop(columns=all_nan_cols)
                cluster_features = [col for col in cluster_features if col not in all_nan_cols]
            
            # Fill remaining NaN with column medians
            X_cluster = X_cluster.fillna(X_cluster.median())
        
        # Verify no NaN values remain
        if X_cluster.isna().any().any():
            logger.error("Still have NaN values after imputation for clustering. Using iterative imputer or dropping rows...")
            # Drop rows with any remaining NaN
            X_cluster = X_cluster.dropna()
            # Also need to update the index of clustering_features
            clustering_features = clustering_features.loc[X_cluster.index]
        
        # Check if we still have enough data
        if X_cluster.shape[0] < n_clusters:
            logger.error(f"Not enough data points ({X_cluster.shape[0]}) for {n_clusters} clusters")
            return pd.DataFrame(index=self.df.index)
        
        if len(cluster_features) < 3:
            logger.warning(f"After cleaning, insufficient features for clustering. Need at least 3, found {len(cluster_features)}")
            return clustering_features
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_cluster)
        
        # Perform K-means clustering
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        
        # Add cluster assignments (matching the original indices)
        clustering_features.loc[X_cluster.index, 'County_Cluster'] = cluster_labels
        
        # Create one-hot encoded cluster features
        cluster_dummies = pd.get_dummies(clustering_features['County_Cluster'], prefix='Cluster')
        clustering_features = pd.concat([clustering_features, cluster_dummies], axis=1)
        
        # Add distance to cluster centers
        distances = kmeans.transform(X_scaled)
        for i in range(n_clusters):
            clustering_features.loc[X_cluster.index, f'Distance_to_Cluster_{i}'] = distances[:, i]
        
        # Fill NaN for counties that weren't included in clustering
        clustering_features = clustering_features.fillna({
            'County_Cluster': -1,
            **{f'Cluster_{i}': 0 for i in range(n_clusters)},
            **{f'Distance_to_Cluster_{i}': 999 for i in range(n_clusters)}
        })
        
        # Identify cluster characteristics
        cluster_centers_df = pd.DataFrame(
            scaler.inverse_transform(kmeans.cluster_centers_),
            columns=cluster_features,
            index=[f'Cluster_{i}' for i in range(n_clusters)]
        )
        
        # Save cluster characteristics for interpretation
        self.cluster_characteristics = cluster_centers_df
        
        logger.info(f"Created {len(clustering_features.columns)} clustering features")
        return clustering_features

    def apply_dimensionality_reduction(self, n_components: int = 10) -> pd.DataFrame:
        """
        Apply PCA for dimensionality reduction and feature extraction.
        
        Args:
            n_components: Number of PCA components to create
            
        Returns:
            DataFrame with PCA components
        """
        logger.info(f"Creating {n_components} PCA components...")
        
        pca_features = pd.DataFrame(index=self.df.index)
        
        # Select numeric features for PCA
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        
        # Remove target columns from PCA
        target_cols = [col for col in self.targets if col in numeric_cols]
        numeric_cols = [col for col in numeric_cols if col not in target_cols]
        
        if len(numeric_cols) < n_components:
            logger.warning(f"Too few features for PCA. Need at least {n_components}, found {len(numeric_cols)}")
            return pca_features
        
        # Prepare data for PCA
        X_pca = self.df[numeric_cols].copy()
        
        # Check for NaN values
        nan_cols = X_pca.columns[X_pca.isna().any()].tolist()
        if nan_cols:
            logger.info(f"Found {len(nan_cols)} columns with NaN values. Filling with column medians...")
            # Calculate medians before filling
            col_medians = X_pca.median()
            
            # Check if any column has all NaN values
            all_nan_cols = X_pca.columns[X_pca.isna().all()].tolist()
            if all_nan_cols:
                logger.warning(f"Columns with all NaN values: {all_nan_cols}. These will be dropped.")
                X_pca = X_pca.drop(columns=all_nan_cols)
                numeric_cols = [col for col in numeric_cols if col not in all_nan_cols]
            
            # Fill remaining NaN with column medians
            X_pca = X_pca.fillna(col_medians)
        
        # Verify no NaN values remain
        remaining_nans = X_pca.isna().sum().sum()
        if remaining_nans > 0:
            logger.error(f"Still have {remaining_nans} NaN values after imputation. Dropping rows with NaN...")
            X_pca = X_pca.dropna()
            # Also need to update the index of pca_features
            pca_features = pca_features.loc[X_pca.index]
        
        # Check if we still have enough data
        if X_pca.shape[0] < 2:
            logger.error("Not enough data points for PCA after NaN removal")
            return pd.DataFrame(index=self.df.index)
        
        if len(X_pca.columns) < n_components:
            logger.warning(f"After cleaning, too few features for PCA. Need at least {n_components}, found {len(X_pca.columns)}")
            n_components = min(n_components, len(X_pca.columns))
        
        # Standardize features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_pca)
        
        # Apply PCA
        pca = PCA(n_components=min(n_components, len(X_pca.columns)))
        pca_components = pca.fit_transform(X_scaled)
        
        # Create DataFrame with PCA components
        for i in range(pca.n_components_):
            pca_features[f'PCA_Component_{i+1}'] = pca_components[:, i]
        
        # Store PCA information for interpretation
        self.pca_explained_variance = pca.explained_variance_ratio_
        self.pca_components_df = pd.DataFrame(
            pca.components_,
            columns=X_pca.columns,
            index=[f'PC{i+1}' for i in range(pca.n_components_)]
        )
        
        logger.info(f"PCA explained variance: {self.pca_explained_variance[:3].sum():.2%} by first 3 components")
        logger.info(f"Created {len(pca_features.columns)} PCA components")
        
        return pca_features
            
    def select_top_features(self, target: str = 'Pct_Diabetes', k: int = 20) -> List[str]:
        """
        Select top k features based on correlation with target.
        
        Args:
            target: Target variable for feature selection
            k: Number of top features to select
            
        Returns:
            List of top feature names
        """
        logger.info(f"Selecting top {k} features for target: {target}")
        
        if target not in self.df.columns:
            logger.error(f"Target {target} not found in dataset")
            return []
        
        # Get all feature columns (excluding the target)
        feature_cols = [col for col in self.df.columns if col != target and pd.api.types.is_numeric_dtype(self.df[col])]
        
        if len(feature_cols) == 0:
            logger.error("No numeric features found for selection")
            return []
        
        # Calculate correlations with target
        correlations = {}
        for col in feature_cols:
            valid_idx = self.df[[target, col]].dropna().index
            if len(valid_idx) > 10:  # Need sufficient data
                corr = self.df.loc[valid_idx, target].corr(self.df.loc[valid_idx, col])
                if not pd.isna(corr):
                    correlations[col] = abs(corr)
        
        # Select top k features
        top_features = sorted(correlations.items(), key=lambda x: x[1], reverse=True)[:k]
        
        logger.info(f"Top 5 features for {target}:")
        for feature, corr in top_features[:5]:
            logger.info(f"  {feature}: {corr:.3f}")
        
        return [feature for feature, _ in top_features]
    
    def create_all_features(self, 
                           include_clustering: bool = True,
                           include_pca: bool = True,
                           n_clusters: int = 5,
                           n_pca_components: int = 10) -> pd.DataFrame:
        """
        Create all feature types and combine them.
        
        Args:
            include_clustering: Whether to include clustering features
            include_pca: Whether to include PCA features
            n_clusters: Number of clusters for K-means
            n_pca_components: Number of PCA components
            
        Returns:
            DataFrame with all engineered features
        """
        logger.info("=" * 70)
        logger.info("STARTING COMPREHENSIVE FEATURE ENGINEERING")
        logger.info("=" * 70)
        
        # Load data if not already loaded
        if self.df is None:
            self.load_data()
            
        # Check data quality first
        self.check_data_quality()
        
        # Initialize list to store all feature DataFrames
        all_features = []
        
        # 1. Clinical Risk Features
        clinical_features = self.create_clinical_risk_features()
        all_features.append(clinical_features)
        self.feature_categories['clinical'] = clinical_features.columns.tolist()
        
        # 2. Behavioral Lifestyle Features
        behavioral_features = self.create_behavioral_lifestyle_features()
        all_features.append(behavioral_features)
        self.feature_categories['behavioral'] = behavioral_features.columns.tolist()
        
        # 3. Environmental Features
        env_features = self.create_environmental_features()
        all_features.append(env_features)
        self.feature_categories['environmental'] = env_features.columns.tolist()
        
        # 4. Demographic Features
        demo_features = self.create_demographic_features()
        all_features.append(demo_features)
        self.feature_categories['demographic'] = demo_features.columns.tolist()
        
        # 5. Composite Risk Scores
        composite_features = self.create_composite_risk_scores()
        all_features.append(composite_features)
        self.feature_categories['composite'] = composite_features.columns.tolist()
        
        # 6. Interaction Features
        interaction_features = self.create_interaction_features()
        all_features.append(interaction_features)
        self.feature_categories['interaction'] = interaction_features.columns.tolist()
        
        # 7. Statistical Features
        stats_features = self.create_statistical_features()
        all_features.append(stats_features)
        self.feature_categories['statistical'] = stats_features.columns.tolist()
        
        # 8. Clustering Features (optional)
        if include_clustering:
            clustering_features = self.create_clustering_features(n_clusters=n_clusters)
            if not clustering_features.empty:
                all_features.append(clustering_features)
                self.feature_categories['clustering'] = clustering_features.columns.tolist()
        
        # 9. PCA Features (optional)
        if include_pca:
            pca_features = self.apply_dimensionality_reduction(n_components=n_pca_components)
            if not pca_features.empty:
                all_features.append(pca_features)
                self.feature_categories['pca'] = pca_features.columns.tolist()
        
        # Combine all features
        self.features = pd.concat(all_features, axis=1)
        
        # Add original county identifier if available
        if 'FIPS' in self.df.columns:
            self.features['FIPS'] = self.df['FIPS']
        if 'State' in self.df.columns:
            self.features['State'] = self.df['State']
        
        # Add sample size for weighting if available
        if 'Sample_Size' in self.df.columns:
            self.features['Sample_Size'] = self.df['Sample_Size']
        
        # IMPORTANT: Add the target columns from the original data
        if self.targets:
            for target in self.targets:
                if target in self.df.columns:
                    self.features[target] = self.df[target]
        
        logger.info("=" * 70)
        logger.info("FEATURE ENGINEERING COMPLETE")
        logger.info("=" * 70)
        logger.info(f"Total engineered features: {len(self.features.columns)}")
        logger.info(f"Feature categories created: {len(self.feature_categories)}")
        
        for category, features in self.feature_categories.items():
            logger.info(f"  {category}: {len(features)} features")
        
        return self.features
    
    def save_features(self, output_path: str = 'data/processed/engineered_features.csv'):
        """
        Save engineered features to CSV file.
        
        Args:
            output_path: Path to save the engineered features
        """
        if self.features is None:
            logger.error("No features to save. Run create_all_features() first.")
            return
        
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        self.features.to_csv(output_path, index=False)
        logger.info(f"Saved engineered features to {output_path}")
        
        # Also save feature categories metadata
        metadata_path = output_path.replace('.csv', '_metadata.json')
        import json
        
        metadata = {
            'total_features': len(self.features.columns),
            'feature_categories': {k: len(v) for k, v in self.feature_categories.items()},
            'dataset_shape': self.features.shape
        }
        
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        logger.info(f"Saved feature metadata to {metadata_path}")
        
        # Save cluster characteristics if available
        if hasattr(self, 'cluster_characteristics'):
            cluster_path = output_path.replace('.csv', '_clusters.csv')
            self.cluster_characteristics.to_csv(cluster_path)
            logger.info(f"Saved cluster characteristics to {cluster_path}")
        
        # Save PCA information if available
        if hasattr(self, 'pca_components_df'):
            pca_path = output_path.replace('.csv', '_pca.csv')
            self.pca_components_df.to_csv(pca_path)
            
            # Save explained variance
            pca_var_path = output_path.replace('.csv', '_pca_variance.csv')
            pd.DataFrame({
                'Component': [f'PC{i+1}' for i in range(len(self.pca_explained_variance))],
                'Explained_Variance': self.pca_explained_variance,
                'Cumulative_Variance': np.cumsum(self.pca_explained_variance)
            }).to_csv(pca_var_path, index=False)
            
            logger.info(f"Saved PCA information to {pca_path} and {pca_var_path}")
    
    def prepare_modeling_data(self, 
                         target_columns: Optional[List[str]] = None,
                         test_size: float = 0.2,
                         random_state: int = 42) -> Dict[str, np.ndarray]:
        """
        Prepare data for modeling with train-test split.
        
        Args:
            target_columns: List of target column names
            test_size: Proportion of data for testing
            random_state: Random seed for reproducibility
            
        Returns:
            Dictionary with X_train, X_test, y_train, y_test arrays
        """
        logger.info("Preparing data for modeling...")
        
        if self.features is None:
            logger.error("No features available. Run create_all_features() first.")
            return {}
        
        # Use default targets if not specified
        if target_columns is None:
            target_columns = self.targets
        
        if not target_columns:
            logger.error("No target columns specified or found")
            return {}
        
        # Identify feature columns (exclude identifiers, targets, and non-numeric columns)
        exclude_cols = ['FIPS', 'State', 'Sample_Size'] + target_columns
        feature_cols = [
            col for col in self.features.columns 
            if col not in exclude_cols 
            and pd.api.types.is_numeric_dtype(self.features[col])
        ]
        
        if not feature_cols:
            logger.error("No numeric feature columns found for modeling")
            return {}
        
        logger.info(f"Using {len(feature_cols)} features for modeling")
        logger.info(f"Predicting {len(target_columns)} targets: {', '.join(target_columns)}")
        
        # Prepare X and y
        X = self.features[feature_cols].copy()
        y = self.features[target_columns].copy()
        
        # Check for NaN values
        x_nan_count = X.isna().sum().sum()
        y_nan_count = y.isna().sum().sum()
        
        if x_nan_count > 0:
            logger.warning(f"Found {x_nan_count} NaN values in features. Filling with column medians...")
            # Fill NaN with median for each column
            for col in X.columns:
                if pd.api.types.is_numeric_dtype(X[col]):
                    X[col] = X[col].fillna(X[col].median())
        
        if y_nan_count > 0:
            logger.warning(f"Found {y_nan_count} NaN values in targets. Filling with column medians...")
            # Fill NaN with median for each target column
            for col in y.columns:
                if pd.api.types.is_numeric_dtype(y[col]):
                    y[col] = y[col].fillna(y[col].median())
        
        # Verify no NaN values remain
        remaining_nans = X.isna().sum().sum() + y.isna().sum().sum()
        if remaining_nans > 0:
            logger.error(f"NaN values still present after imputation: {remaining_nans}. Dropping rows with NaN...")
            combined = pd.concat([X, y], axis=1)
            combined = combined.dropna()
            X = combined[feature_cols]
            y = combined[target_columns]
        
        logger.info(f"Final dataset shape: X={X.shape}, y={y.shape}")
        
        # Check if we have enough data
        if X.shape[0] < 10:
            logger.error(f"Not enough data points ({X.shape[0]}) for train-test split")
            return {}
        
        # Train-test split
        from sklearn.model_selection import train_test_split
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        logger.info(f"Train set: {X_train.shape}")
        logger.info(f"Test set: {X_test.shape}")
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        self.scalers['feature_scaler'] = scaler
        
        return {
            'X_train': X_train_scaled,
            'X_test': X_test_scaled,
            'y_train': y_train.values,
            'y_test': y_test.values,
            'feature_names': feature_cols,
            'target_names': target_columns,
            'X_train_raw': X_train.values,
            'X_test_raw': X_test.values
        }

# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    """Main function to execute feature engineering pipeline."""
    logger.info("=" * 70)
    logger.info("MULTI-DISEASE PREDICTION: FEATURE ENGINEERING PIPELINE")
    logger.info("=" * 70)
    
    # Configuration
    INPUT_DATA = '../data/processed/multi_disease_final_dataset_2023.csv'
    OUTPUT_FEATURES = 'data/processed/engineered_features.csv'
    
    # Create output directories
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('reports/feature_analysis', exist_ok=True)
    
    try:
        # Initialize feature engineer
        engineer = MultiDiseaseFeatureEngineer(INPUT_DATA)
        
        # Load data
        engineer.load_data()
        
        # Create all features
        features_df = engineer.create_all_features(
            include_clustering=True,
            include_pca=True,
            n_clusters=6,
            n_pca_components=12
        )
        
        # Save engineered features
        engineer.save_features(OUTPUT_FEATURES)
        
        # Prepare data for modeling
        modeling_data = engineer.prepare_modeling_data(
            target_columns=['Pct_Diabetes', 'Pct_HeartDisease', 'Pct_Stroke', 'Pct_Asthma'],
            test_size=0.2,
            random_state=42
        )
        
        if modeling_data:
            logger.info("\n" + "=" * 70)
            logger.info("MODELING DATA PREPARED SUCCESSFULLY")
            logger.info("=" * 70)
            logger.info(f"Training samples: {modeling_data['X_train'].shape[0]}")
            logger.info(f"Testing samples: {modeling_data['X_test'].shape[0]}")
            logger.info(f"Number of features: {modeling_data['X_train'].shape[1]}")
            logger.info(f"Number of targets: {len(modeling_data['target_names'])}")
            
            # Save modeling data for later use
            import joblib
            joblib.dump(modeling_data, 'data/processed/modeling_data.pkl')
            logger.info("Saved modeling data to data/processed/modeling_data.pkl")
            
            # Generate feature importance report
            generate_feature_importance_report(engineer, modeling_data)
        else:
            logger.warning("No modeling data was prepared. Skipping feature importance report.")
        
        logger.info("\n" + "=" * 70)
        logger.info("FEATURE ENGINEERING PIPELINE COMPLETE")
        logger.info("=" * 70)
        logger.info(f"Input data: {INPUT_DATA}")
        logger.info(f"Output features: {OUTPUT_FEATURES}")
        logger.info(f"Total engineered features: {len(features_df.columns)}")
        
    except Exception as e:
        logger.error(f"Feature engineering pipeline failed: {e}")
        import traceback
        traceback.print_exc()

def generate_feature_importance_report(engineer: MultiDiseaseFeatureEngineer, 
                                      modeling_data: Dict[str, np.ndarray]):
    """Generate a report on feature importance."""
    logger.info("Generating feature importance report...")
    
    report_dir = 'reports/feature_analysis'
    os.makedirs(report_dir, exist_ok=True)
    
    # Check if modeling_data is valid
    if not modeling_data:
        logger.warning("No modeling data available for feature importance report")
        return
    
    if 'feature_names' not in modeling_data or not modeling_data['feature_names']:
        logger.warning("No feature names available for feature importance report")
        return
    
    if 'target_names' not in modeling_data or not modeling_data['target_names']:
        logger.warning("No target names available for feature importance report")
        return
    
    if 'X_train_raw' not in modeling_data or 'y_train' not in modeling_data:
        logger.warning("Training data not available for feature importance report")
        return
    
    feature_names = modeling_data['feature_names']
    
    # Calculate correlation with each target
    importance_data = []
    
    for target_idx, target_name in enumerate(modeling_data['target_names']):
        # Get target values
        y_target = modeling_data['y_train'][:, target_idx]
        
        # Calculate correlation with each feature
        for feat_idx, feat_name in enumerate(feature_names):
            X_feature = modeling_data['X_train_raw'][:, feat_idx]
            
            # Calculate correlation
            corr = np.corrcoef(X_feature, y_target)[0, 1]
            if not np.isnan(corr):
                importance_data.append({
                    'Target': target_name,
                    'Feature': feat_name,
                    'Correlation': abs(corr),
                    'Correlation_Signed': corr
                })
    
    # Create DataFrame and sort by importance
    importance_df = pd.DataFrame(importance_data)
    
    if importance_df.empty:
        logger.warning("No correlation data calculated for feature importance report")
        return
    
    # Save detailed importance data
    importance_df.to_csv(f'{report_dir}/feature_importance_detailed.csv', index=False)
    
    # Create summary report
    summary_report = f"""
    FEATURE IMPORTANCE ANALYSIS REPORT
    ===================================
    
    Dataset Overview:
    - Total features: {len(feature_names)}
    - Targets: {', '.join(modeling_data['target_names'])}
    - Training samples: {modeling_data['X_train'].shape[0] if 'X_train' in modeling_data else 'N/A'}
    
    Top Features by Target:
    """
    
    for target in modeling_data['target_names']:
        target_data = importance_df[importance_df['Target'] == target]
        if target_data.empty:
            summary_report += f"\n\n{target}:\n"
            summary_report += "-" * 50 + "\n"
            summary_report += "No feature importance data available\n"
            continue
            
        top_features = target_data.nlargest(10, 'Correlation')
        
        summary_report += f"\n\n{target}:\n"
        summary_report += "-" * 50 + "\n"
        
        for _, row in top_features.iterrows():
            summary_report += f"{row['Feature']}: {row['Correlation_Signed']:.3f}\n"
    
    # Save summary report
    with open(f'{report_dir}/feature_importance_summary.txt', 'w') as f:
        f.write(summary_report)
    
    logger.info(f"Feature importance reports saved to {report_dir}/")
    
    # Create visualization of top features
    try:
        import matplotlib.pyplot as plt
        
        plt.figure(figsize=(12, 8))
        
        for i, target in enumerate(modeling_data['target_names'][:4]):  # Show first 4 targets
            plt.subplot(2, 2, i + 1)
            
            target_data = importance_df[importance_df['Target'] == target]
            if target_data.empty:
                plt.text(0.5, 0.5, f'No data for {target}', 
                        ha='center', va='center', transform=plt.gca().transAxes)
                plt.title(f'No data for {target}')
                continue
                
            top_features = target_data.nlargest(8, 'Correlation')
            
            colors = ['green' if x > 0 else 'red' for x in top_features['Correlation_Signed']]
            plt.barh(range(len(top_features)), top_features['Correlation_Signed'], color=colors)
            plt.yticks(range(len(top_features)), top_features['Feature'], fontsize=8)
            plt.title(f'Top Features for {target}')
            plt.xlabel('Correlation')
            plt.tight_layout()
        
        plt.savefig(f'{report_dir}/top_features_visualization.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        logger.info(f"Feature visualization saved to {report_dir}/top_features_visualization.png")
        
    except ImportError:
        logger.warning("Matplotlib not available. Skipping visualization.")
    
    except Exception as e:
        logger.warning(f"Could not create visualization: {e}")


if __name__ == "__main__":
    main()

2025-12-14 15:52:43,967 - INFO - ======================================================================
2025-12-14 15:52:43,968 - INFO - MULTI-DISEASE PREDICTION: FEATURE ENGINEERING PIPELINE
2025-12-14 15:52:43,968 - INFO - ======================================================================
2025-12-14 15:52:43,969 - INFO - Loading data from ../data/processed/multi_disease_final_dataset_2023.csv
2025-12-14 15:52:43,972 - INFO - Loaded dataset: 52 counties, 41 features
2025-12-14 15:52:43,973 - INFO - Found 18 columns with pattern 'Pct_'
2025-12-14 15:52:43,973 - INFO - Found 9 columns with pattern 'Avg_'
2025-12-14 15:52:43,973 - INFO - Found 5 columns with pattern 'State_'
2025-12-14 15:52:43,974 - INFO - Found columns with patterns: Pct_, Avg_, State_
2025-12-14 15:52:43,974 - INFO - Found 4 disease targets with valid data: Pct_Diabetes, Pct_HeartDisease, Pct_Stroke, Pct_Asthma
2025-12-14 15:52:43,975 - INFO -   Pct_Diabetes: 52 valid values, mean=14.24, range=[9.30, 20.42]
2025-1